# Nano PSM Retention Decay From Reviewed Checkpoint

Fine-tune the primary 10M Nano PSM model on the retention-decay curriculum, initialized from the reviewed-5k best checkpoint.

In [ ]:
!pip install -U huggingface_hub datasets safetensors onnx onnxruntime
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
HF_DATASET_REPO = 'chkrishna2001/nano-psm-retention-decay-5k'
HF_PREVIOUS_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-reviewed-5k-checkpoints'
HF_OUTPUT_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-retention-decay-from-reviewed-checkpoints'
CODE_REPO = 'https://github.com/chkrishna2001/PSM.git'
CODE_REVISION = 'main'

LOCAL_DATA_DIR = '/content/psm-retention-decay-5k'
LOCAL_PREVIOUS_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-reviewed-5k-checkpoints'
LOCAL_OUTPUT_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-retention-decay-from-reviewed-checkpoints'
REPO_DIR = '/content/PSM'


## Download Dataset, Code, And Reviewed Checkpoint

In [ ]:
from huggingface_hub import snapshot_download

dataset_dir = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    local_dir=LOCAL_DATA_DIR,
    force_download=True,
)
print('dataset:', dataset_dir)

previous_checkpoint_dir = snapshot_download(
    repo_id=HF_PREVIOUS_CHECKPOINT_REPO,
    repo_type='model',
    local_dir=LOCAL_PREVIOUS_CHECKPOINT_DIR,
    force_download=True,
)
print('reviewed checkpoint:', previous_checkpoint_dir)

try:
    output_checkpoint_dir = snapshot_download(
        repo_id=HF_OUTPUT_CHECKPOINT_REPO,
        repo_type='model',
        local_dir=LOCAL_OUTPUT_CHECKPOINT_DIR,
        force_download=True,
    )
    print('resume checkpoint:', output_checkpoint_dir)
except Exception as exc:
    output_checkpoint_dir = LOCAL_OUTPUT_CHECKPOINT_DIR
    print('No retention checkpoint snapshot yet; will initialize from reviewed checkpoint:', exc)

!git clone {CODE_REPO} {REPO_DIR} || true
%cd {REPO_DIR}
!git fetch origin
!git checkout {CODE_REVISION}
!git pull --ff-only || true
!git rev-parse --short HEAD

## Preflight

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

train_path = Path(LOCAL_DATA_DIR) / 'train.jsonl'
validation_path = Path(LOCAL_DATA_DIR) / 'validation.jsonl'
metadata_path = Path(LOCAL_DATA_DIR) / 'metadata.json'
config_path = Path('nano-psm/configs/primary-10m.json')
init_checkpoint_path = Path(LOCAL_PREVIOUS_CHECKPOINT_DIR) / 'checkpoint-best.pt'
for required in (train_path, validation_path, metadata_path, config_path, init_checkpoint_path):
    assert required.exists(), f'Missing required file: {required}'

sys.path.insert(0, 'nano-psm/src')
from nano_psm.model import NanoPsmConfig
from nano_psm.dataset import load_jsonl

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
config_doc = json.loads(config_path.read_text(encoding='utf-8'))
model_config = NanoPsmConfig(**config_doc['model'])
train_examples = load_jsonl(train_path)
validation_examples = load_jsonl(validation_path)

print('dataset', HF_DATASET_REPO)
print('previous_checkpoint', HF_PREVIOUS_CHECKPOINT_REPO)
print('init_checkpoint', init_checkpoint_path)
print('output_checkpoint', HF_OUTPUT_CHECKPOINT_REPO)
print('metadata_total_examples', metadata.get('total_examples'))
print('train_examples', len(train_examples))
print('validation_examples', len(validation_examples))
print('action_mix', metadata.get('action_mix'))
print('source_mix', metadata.get('source_mix'))
print('retention_mix', metadata.get('retention_mix'))
print('parameter_estimate', f'{model_config.estimate_parameters():,}')
assert len(train_examples) == 4285
assert len(validation_examples) == 715
assert metadata.get('total_examples') == 5000
assert 8_000_000 <= model_config.estimate_parameters() <= 12_000_000

subprocess.run([sys.executable, '-m', 'compileall', 'nano-psm/src/nano_psm'], check=True)
print('preflight ok')

## Train Retention Curriculum

In [ ]:
MAX_STEPS = 4000
EVAL_EVERY = 250
SAVE_EVERY = 500

!python nano-psm/src/nano_psm/train.py \
  --config nano-psm/configs/primary-10m.json \
  --train {LOCAL_DATA_DIR}/train.jsonl \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint-dir {LOCAL_OUTPUT_CHECKPOINT_DIR} \
  --init-checkpoint {LOCAL_PREVIOUS_CHECKPOINT_DIR}/checkpoint-best.pt \
  --resume auto \
  --device auto \
  --max-steps {MAX_STEPS} \
  --eval-every {EVAL_EVERY} \
  --save-every {SAVE_EVERY}

## Evaluate Retention Validation

In [ ]:
!python nano-psm/src/nano_psm/evaluate.py \
  --config nano-psm/configs/primary-10m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_OUTPUT_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto

In [ ]:
from google.colab import files
from pathlib import Path

out = Path('/content/validationresults-retention-decay-from-reviewed.json')
!python nano-psm/src/nano_psm/inspect_predictions.py \
  --config nano-psm/configs/primary-10m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_OUTPUT_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto \
  --limit 100000 \
  > /content/validationresults-retention-decay-from-reviewed.json
print('written:', out, 'bytes:', out.stat().st_size)
files.download(str(out))

## Upload Retention Checkpoints

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(repo_id=HF_OUTPUT_CHECKPOINT_REPO, repo_type='model', private=True, exist_ok=True)
upload_folder(
    repo_id=HF_OUTPUT_CHECKPOINT_REPO,
    repo_type='model',
    folder_path=LOCAL_OUTPUT_CHECKPOINT_DIR,
    path_in_repo='.',
    commit_message='sync nano psm retention decay checkpoint from reviewed base'
)